# Advanced Pandas Practice Notebook

This notebook covers advanced pandas features and techniques to take your skills to the next level. Each section includes explanations, examples, and exercises.

**Topics Covered:**
1. MultiIndex & Advanced Indexing
2. GroupBy — transform, filter, apply, named aggregations
3. Window Functions — rolling, expanding, ewm
4. Reshaping — pivot_table, melt, stack/unstack, crosstab
5. Advanced Merging — merge_asof, cross merge, indicator
6. Method Chaining & pipe
7. eval() & query() for Performance
8. Categorical Data
9. Advanced DateTime Operations
10. Advanced String Operations
11. Memory Optimization
12. Vectorization & Performance Patterns
13. Custom Accessors
14. Styling DataFrames

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

pandas version: 2.3.3
numpy version: 2.2.6


---
## 1. MultiIndex & Advanced Indexing

MultiIndex (hierarchical indexing) allows you to have multiple levels of index on rows or columns. This is essential for working with higher-dimensional data in a 2D DataFrame.

In [ ]:
# Creating a MultiIndex DataFrame
arrays = [
    ['Electronics', 'Electronics', 'Electronics', 'Clothing', 'Clothing', 'Clothing'],
    ['Laptop', 'Phone', 'Tablet', 'Shirt', 'Pants', 'Jacket']
]
index = pd.MultiIndex.from_arrays(arrays, names=['Category', 'Product'])

df_multi = pd.DataFrame({
    'Q1_Sales': [150, 300, 80, 200, 120, 60],
    'Q2_Sales': [180, 350, 95, 220, 140, 75],
    'Q3_Sales': [200, 280, 110, 190, 160, 90],
    'Q4_Sales': [220, 400, 130, 250, 180, 100],
    'Price': [999, 699, 449, 39, 59, 129]
}, index=index)

df_multi

MultiIndex([('Electronics', 'Laptop'),
            ('Electronics',  'Phone'),
            ('Electronics', 'Tablet'),
            (   'Clothing',  'Shirt'),
            (   'Clothing',  'Pants'),
            (   'Clothing', 'Jacket')],
           names=['Category', 'Product'])


Q1_Sales  Q2_Sales  Q3_Sales  Q4_Sales  Price
Category    Product                                               
Electronics Laptop        150       180       200       220    999
            Phone         300       350       280       400    699
            Tablet         80        95       110       130    449
Clothing    Shirt         200       220       190       250     39
            Pants         120       140       160       180     59
            Jacket         60        75        90       100    129

In [ ]:
# xs() — cross-section selection across levels
# Select all products in 'Electronics'
print("All Electronics:")
print(df_multi.xs('Electronics', level='Category'))

print("\nSelect 'Laptop' across categories (level=1):")
print(df_multi.xs('Laptop', level='Product'))

In [ ]:
# IndexSlice — more readable slicing on MultiIndex
idx = pd.IndexSlice

# Slice specific categories and columns
print("Electronics, Q1 and Q2 sales:")
print(df_multi.loc[idx['Electronics', :], ['Q1_Sales', 'Q2_Sales']])

print("\nAll categories, only 'Laptop' and 'Shirt':")
print(df_multi.loc[idx[:, ['Laptop', 'Shirt']], :])

In [ ]:
# swaplevel + sort_index — rearrange hierarchy
df_swapped = df_multi.swaplevel().sort_index()
print("Swapped levels (Product → Category):")
print(df_swapped.head())

In [ ]:
# MultiIndex on columns
col_arrays = [
    ['Revenue', 'Revenue', 'Cost', 'Cost'],
    ['Online', 'Store', 'Online', 'Store']
]
col_index = pd.MultiIndex.from_arrays(col_arrays, names=['Metric', 'Channel'])

df_col_multi = pd.DataFrame(
    np.random.randint(100, 1000, (4, 4)),
    index=['Q1', 'Q2', 'Q3', 'Q4'],
    columns=col_index
)
print("MultiIndex columns:")
print(df_col_multi)

print("\nAccess 'Revenue' across all channels:")
print(df_col_multi['Revenue'])

### Exercise 1
Using `df_multi` above:
1. Compute the total yearly sales (sum of Q1–Q4) per product and add it as a new column.
2. Use `groupby(level='Category')` to find the category with the highest average price.
3. Use `xs` to extract only rows where the product is 'Phone'.

In [ ]:
# Solution 1.1: Total yearly sales per product
sales_cols = ['Q1_Sales', 'Q2_Sales', 'Q3_Sales', 'Q4_Sales']
df_multi['Total_Sales'] = df_multi[sales_cols].sum(axis=1)
print("1. Total yearly sales:")
print(df_multi[['Total_Sales']])

# Solution 1.2: Category with highest average price
print("\n2. Average price by category:")
avg_price = df_multi.groupby(level='Category')['Price'].mean()
print(avg_price)
print(f"\nHighest avg price category: {avg_price.idxmax()} (${avg_price.max():.2f})")

# Solution 1.3: Extract rows where product is 'Phone'
print("\n3. Phone rows using xs:")
print(df_multi.xs('Phone', level='Product'))


---
## 2. GroupBy — Advanced Operations

Beyond basic `groupby().agg()`, pandas offers `transform`, `filter`, `apply`, and **named aggregations** for powerful group-level computations.

In [ ]:
np.random.seed(42)
df_sales = pd.DataFrame({
    'region': np.random.choice(['North', 'South', 'East', 'West'], 200),
    'product': np.random.choice(['A', 'B', 'C'], 200),
    'salesperson': np.random.choice(['Alice', 'Bob', 'Charlie', 'Diana'], 200),
    'revenue': np.random.normal(1000, 300, 200).round(2),
    'units': np.random.randint(1, 50, 200),
    'date': pd.date_range('2024-01-01', periods=200, freq='D')
})
df_sales.head()

In [ ]:
# transform — returns a Series with the same index as the original DataFrame.
# Useful for broadcasting group-level statistics back to each row.

# Normalize revenue within each region (z-score)
df_sales['revenue_zscore'] = df_sales.groupby('region')['revenue'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Percentage of group total
df_sales['revenue_pct_of_region'] = df_sales.groupby('region')['revenue'].transform(
    lambda x: x / x.sum() * 100
)

df_sales[['region', 'revenue', 'revenue_zscore', 'revenue_pct_of_region']].head(10)

In [ ]:
# filter — keep only groups that satisfy a condition
# Keep only regions where average revenue > 1000
high_revenue_regions = df_sales.groupby('region').filter(
    lambda g: g['revenue'].mean() > 1000
)
print(f"Original rows: {len(df_sales)}, After filter: {len(high_revenue_regions)}")
print("Remaining regions:", high_revenue_regions['region'].unique())

In [ ]:
# Named aggregations — clean syntax for multiple aggregations with custom names
result = df_sales.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    median_units=('units', 'median'),
    max_units=('units', 'max'),
    num_transactions=('revenue', 'count'),
    revenue_std=('revenue', 'std')
)
result

In [ ]:
# apply — for complex per-group logic that returns arbitrary shapes
# Top 2 revenue transactions per region
top2_per_region = df_sales.groupby('region').apply(
    lambda g: g.nlargest(2, 'revenue')[['product', 'revenue', 'salesperson']],
    include_groups=False
)
top2_per_region

In [ ]:
# Cumulative operations within groups
df_sales_sorted = df_sales.sort_values(['region', 'date'])
df_sales_sorted['cumulative_revenue'] = df_sales_sorted.groupby('region')['revenue'].cumsum()
df_sales_sorted['running_avg'] = df_sales_sorted.groupby('region')['revenue'].expanding().mean().reset_index(level=0, drop=True)

df_sales_sorted[['region', 'date', 'revenue', 'cumulative_revenue', 'running_avg']].head(10)

In [ ]:
# Groupby with multiple keys + unstack for pivot-like result
region_product = df_sales.groupby(['region', 'product'])['revenue'].mean().unstack(fill_value=0)
region_product

### Exercise 2
1. Use `transform` to create a column showing the rank of each transaction's revenue within its region.
2. Use `filter` to keep only products where total units sold > 300.
3. Write a named aggregation that computes, per `(region, product)`: min revenue, max revenue, and total units.

In [ ]:
# Solution 2.1: Rank of revenue within each region using transform
df_sales['revenue_rank_in_region'] = df_sales.groupby('region')['revenue'].transform(
    lambda x: x.rank(ascending=False)
)
print("1. Revenue rank within region:")
print(df_sales[['region', 'revenue', 'revenue_rank_in_region']].head(10))

# Solution 2.2: Keep only products where total units > 300
products_above_300 = df_sales.groupby('product').filter(
    lambda g: g['units'].sum() > 300
)
print(f"\n2. Original rows: {len(df_sales)}, After filter: {len(products_above_300)}")
print("Remaining products:", products_above_300['product'].unique())
print("Units per product:")
print(df_sales.groupby('product')['units'].sum())

# Solution 2.3: Named aggregation per (region, product)
region_product_agg = df_sales.groupby(['region', 'product']).agg(
    min_revenue=('revenue', 'min'),
    max_revenue=('revenue', 'max'),
    total_units=('units', 'sum')
)
print("\n3. Named aggregation (region × product):")
region_product_agg


---
## 3. Window Functions — rolling, expanding, ewm

Window functions compute statistics over a sliding or expanding window of data. Critical for time-series analysis.

In [ ]:
np.random.seed(99)
dates = pd.date_range('2024-01-01', periods=120, freq='D')
stock = pd.DataFrame({
    'date': dates,
    'price': 100 + np.cumsum(np.random.randn(120) * 2),
    'volume': np.random.randint(1000, 10000, 120)
}).set_index('date')

stock.head()

In [ ]:
# Rolling — fixed-size sliding window
stock['sma_7'] = stock['price'].rolling(window=7).mean()
stock['sma_30'] = stock['price'].rolling(window=30).mean()
stock['rolling_std_7'] = stock['price'].rolling(window=7).std()

# Bollinger Bands
stock['upper_band'] = stock['sma_7'] + 2 * stock['rolling_std_7']
stock['lower_band'] = stock['sma_7'] - 2 * stock['rolling_std_7']

stock[['price', 'sma_7', 'sma_30', 'upper_band', 'lower_band']].tail(10)

In [ ]:
# Rolling with custom function
# Max drawdown in a 14-day window
stock['max_drawdown_14'] = stock['price'].rolling(14).apply(
    lambda w: (w.min() - w.iloc[0]) / w.iloc[0] * 100,
    raw=False
)
stock[['price', 'max_drawdown_14']].tail(10)

In [ ]:
# Expanding — cumulative window from start
stock['expanding_mean'] = stock['price'].expanding().mean()
stock['expanding_max'] = stock['price'].expanding().max()
stock['expanding_min'] = stock['price'].expanding().min()

stock[['price', 'expanding_mean', 'expanding_max', 'expanding_min']].tail(10)

In [ ]:
# EWM — Exponentially Weighted Moving average
# Gives more weight to recent observations
stock['ewm_12'] = stock['price'].ewm(span=12, adjust=False).mean()
stock['ewm_26'] = stock['price'].ewm(span=26, adjust=False).mean()

# MACD (Moving Average Convergence Divergence)
stock['macd'] = stock['ewm_12'] - stock['ewm_26']
stock['signal_line'] = stock['macd'].ewm(span=9, adjust=False).mean()

stock[['price', 'ewm_12', 'ewm_26', 'macd', 'signal_line']].tail(10)

In [ ]:
# Rolling correlation between price and volume
stock['price_vol_corr_14'] = stock['price'].rolling(14).corr(stock['volume'])
stock[['price', 'volume', 'price_vol_corr_14']].tail(10)

### Exercise 3
1. Compute a 21-day rolling median of `price`.
2. Compute the rolling 7-day percentage change: `(current_price - price_7_days_ago) / price_7_days_ago * 100`. Hint: use `.shift()`.
3. Use `ewm` with `halflife=5` to create a smoothed volume series.

In [ ]:
# Solution 3.1: 21-day rolling median of price
stock['rolling_median_21'] = stock['price'].rolling(window=21).median()
print("1. 21-day rolling median:")
print(stock[['price', 'rolling_median_21']].tail(10))

# Solution 3.2: Rolling 7-day percentage change using shift
stock['pct_change_7d'] = (
    (stock['price'] - stock['price'].shift(7)) / stock['price'].shift(7) * 100
)
print("\n2. 7-day percentage change:")
print(stock[['price', 'pct_change_7d']].tail(10))

# Solution 3.3: EWM with halflife=5 for smoothed volume
stock['smoothed_volume'] = stock['volume'].ewm(halflife=5).mean()
print("\n3. EWM smoothed volume (halflife=5):")
print(stock[['volume', 'smoothed_volume']].tail(10))


---
## 4. Reshaping — pivot_table, melt, stack/unstack, crosstab

Reshaping transforms the structure of your data between long and wide formats.

In [ ]:
np.random.seed(42)
df_long = pd.DataFrame({
    'store': np.repeat(['Store_A', 'Store_B', 'Store_C'], 12),
    'month': np.tile(pd.date_range('2024-01', periods=12, freq='MS'), 3),
    'product': np.random.choice(['Widget', 'Gadget', 'Doohickey'], 36),
    'sales': np.random.randint(50, 500, 36),
    'returns': np.random.randint(0, 30, 36)
})
df_long.head(10)

In [ ]:
# pivot_table — aggregation + reshaping in one step
# margins=True adds row/column totals
pivot = df_long.pivot_table(
    values='sales',
    index='store',
    columns='product',
    aggfunc=['mean', 'sum'],
    margins=True,
    margins_name='Total'
)
pivot

In [ ]:
# melt — wide to long (opposite of pivot)
df_wide = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie'],
    'math_score': [85, 92, 78],
    'science_score': [90, 88, 95],
    'english_score': [78, 85, 82]
})

df_melted = df_wide.melt(
    id_vars='student',
    value_vars=['math_score', 'science_score', 'english_score'],
    var_name='subject',
    value_name='score'
)
print("Wide format:")
print(df_wide)
print("\nLong format (melted):")
print(df_melted)

In [ ]:
# stack / unstack — reshaping with MultiIndex
# stack() pivots columns into index level, unstack() does the reverse
stacked = df_wide.set_index('student')[['math_score', 'science_score']].stack()
print("Stacked (columns → index level):")
print(stacked)

print("\nUnstacked back (index level → columns):")
print(stacked.unstack())

In [ ]:
# crosstab — frequency table / contingency table
ct = pd.crosstab(
    df_long['store'],
    df_long['product'],
    values=df_long['sales'],
    aggfunc='sum',
    normalize='index'  # normalize by row (shows proportions per store)
)
print("Crosstab (normalized by store):")
ct

### Exercise 4
1. Create a pivot table of `df_long` with `month` as index, `store` as columns, and `returns` as values (sum).
2. Melt the result back to long format.
3. Create a crosstab of `store` vs `product` showing the count of transactions, with margins.

In [ ]:
# Solution 4.1: Pivot table — month as index, store as columns, sum of returns
pivot_returns = df_long.pivot_table(
    values='returns',
    index='month',
    columns='store',
    aggfunc='sum'
)
print("1. Pivot table (returns by month × store):")
print(pivot_returns.head())

# Solution 4.2: Melt the pivot back to long format
melted_back = pivot_returns.reset_index().melt(
    id_vars='month',
    value_vars=['Store_A', 'Store_B', 'Store_C'],
    var_name='store',
    value_name='total_returns'
)
print("\n2. Melted back to long format:")
print(melted_back.head(10))

# Solution 4.3: Crosstab of store vs product (count) with margins
ct_count = pd.crosstab(
    df_long['store'],
    df_long['product'],
    margins=True,
    margins_name='Total'
)
print("\n3. Crosstab (count with margins):")
print(ct_count)


---
## 5. Advanced Merging — merge_asof, cross merge, indicator

Beyond standard inner/outer/left/right joins, pandas offers specialized merge operations.

In [ ]:
# merge with indicator — see which rows matched
left = pd.DataFrame({'key': ['A', 'B', 'C', 'D'], 'value_l': [1, 2, 3, 4]})
right = pd.DataFrame({'key': ['B', 'C', 'D', 'E'], 'value_r': [20, 30, 40, 50]})

merged = left.merge(right, on='key', how='outer', indicator=True)
print("Merge with indicator:")
print(merged)
print("\nValue counts of _merge:")
print(merged['_merge'].value_counts())

In [ ]:
# merge_asof — nearest-match merge (great for time series)
# Matches each row in left with the closest preceding row in right
trades = pd.DataFrame({
    'time': pd.to_datetime(['2024-01-01 10:00:01', '2024-01-01 10:00:03',
                            '2024-01-01 10:00:05', '2024-01-01 10:00:08']),
    'ticker': ['AAPL', 'GOOG', 'AAPL', 'GOOG'],
    'quantity': [100, 200, 150, 50]
})

quotes = pd.DataFrame({
    'time': pd.to_datetime(['2024-01-01 10:00:00', '2024-01-01 10:00:02',
                            '2024-01-01 10:00:04', '2024-01-01 10:00:06']),
    'ticker': ['AAPL', 'AAPL', 'GOOG', 'GOOG'],
    'bid': [150.0, 150.5, 2800.0, 2805.0],
    'ask': [150.1, 150.6, 2800.5, 2805.5]
})

# Match each trade with the most recent quote for the same ticker
asof_merged = pd.merge_asof(
    trades.sort_values('time'),
    quotes.sort_values('time'),
    on='time',
    by='ticker',
    direction='backward'  # look backward in time
)
print("merge_asof result:")
asof_merged

In [ ]:
# Cross merge — cartesian product
sizes = pd.DataFrame({'size': ['S', 'M', 'L', 'XL']})
colors = pd.DataFrame({'color': ['Red', 'Blue', 'Green']})

all_combinations = sizes.merge(colors, how='cross')
print(f"Cross merge: {len(sizes)} × {len(colors)} = {len(all_combinations)} combinations")
all_combinations

In [ ]:
# combine_first — fill NaN in one DataFrame with values from another
df1 = pd.DataFrame({'A': [1, np.nan, 3], 'B': [np.nan, 5, 6]}, index=[0, 1, 2])
df2 = pd.DataFrame({'A': [10, 20, 30], 'B': [40, 50, 60]}, index=[0, 1, 2])

combined = df1.combine_first(df2)
print("df1:")
print(df1)
print("\ndf2:")
print(df2)
print("\ndf1.combine_first(df2) — fills df1's NaNs with df2's values:")
print(combined)

### Exercise 5
1. Create two DataFrames with overlapping and non-overlapping keys. Merge them with `indicator=True` and filter rows that only exist in the left DataFrame.
2. Create a trades DataFrame and a quotes DataFrame with timestamps. Use `merge_asof` with a `tolerance` of `pd.Timedelta('2s')` to only match quotes within 2 seconds.

In [ ]:
# Solution 5.1: Merge with indicator, filter left-only rows
df_left = pd.DataFrame({'id': [1, 2, 3, 4, 5], 'name': ['A', 'B', 'C', 'D', 'E']})
df_right = pd.DataFrame({'id': [3, 4, 5, 6, 7], 'score': [80, 90, 70, 85, 95]})

merged_ind = df_left.merge(df_right, on='id', how='outer', indicator=True)
print("1. Full merge with indicator:")
print(merged_ind)

left_only = merged_ind[merged_ind['_merge'] == 'left_only']
print("\nRows only in left DataFrame:")
print(left_only)

# Solution 5.2: merge_asof with tolerance of 2 seconds
my_trades = pd.DataFrame({
    'time': pd.to_datetime([
        '2024-01-01 10:00:01',
        '2024-01-01 10:00:04',
        '2024-01-01 10:00:10'
    ]),
    'ticker': ['AAPL', 'AAPL', 'AAPL'],
    'qty': [100, 200, 150]
})

my_quotes = pd.DataFrame({
    'time': pd.to_datetime([
        '2024-01-01 10:00:00',
        '2024-01-01 10:00:03',
        '2024-01-01 10:00:06'
    ]),
    'ticker': ['AAPL', 'AAPL', 'AAPL'],
    'price': [150.0, 151.0, 152.0]
})

asof_result = pd.merge_asof(
    my_trades.sort_values('time'),
    my_quotes.sort_values('time'),
    on='time',
    by='ticker',
    tolerance=pd.Timedelta('2s'),
    direction='backward'
)
print("\n2. merge_asof with 2s tolerance:")
print(asof_result)
print("Note: the 10:00:10 trade has NaN price — no quote within 2 seconds.")


---
## 6. Method Chaining & pipe

Method chaining lets you compose multiple operations in a readable pipeline. `pipe()` integrates custom functions into the chain.

In [ ]:
np.random.seed(7)
raw_data = pd.DataFrame({
    'name': ['  Alice ', 'BOB', ' charlie', 'DIANA  ', 'Eve', None, 'Frank'],
    'department': ['Sales', 'sales', 'Engineering', 'ENGINEERING', 'Sales', 'HR', 'hr'],
    'salary': [75000, 82000, 95000, 110000, np.nan, 65000, 70000],
    'tenure_years': [3, 5, 7, 10, 2, np.nan, 4],
    'rating': [4.2, 3.8, 4.5, 4.9, 3.5, 4.0, np.nan]
})

raw_data

In [ ]:
def clean_strings(df, columns):
    """Strip whitespace and title-case string columns."""
    for col in columns:
        df[col] = df[col].str.strip().str.title()
    return df

def add_salary_band(df, col='salary'):
    """Bin salaries into bands."""
    df['salary_band'] = pd.cut(
        df[col],
        bins=[0, 70000, 90000, 120000],
        labels=['Junior', 'Mid', 'Senior']
    )
    return df

# Chained pipeline
result = (
    raw_data
    .dropna(subset=['name'])
    .pipe(clean_strings, columns=['name', 'department'])
    .assign(
        salary=lambda df: df['salary'].fillna(df['salary'].median()),
        tenure_years=lambda df: df['tenure_years'].fillna(df['tenure_years'].median()),
        rating=lambda df: df['rating'].fillna(df['rating'].mean())
    )
    .pipe(add_salary_band)
    .sort_values('salary', ascending=False)
    .reset_index(drop=True)
)
result

In [ ]:
# pipe with arguments — useful for reusable transformations
def filter_by_quantile(df, column, lower=0.1, upper=0.9):
    """Keep rows within the given quantile range."""
    lo = df[column].quantile(lower)
    hi = df[column].quantile(upper)
    return df.query(f"{column} >= @lo and {column} <= @hi")

filtered = result.pipe(filter_by_quantile, column='salary', lower=0.2, upper=0.8)
print(f"Before filter: {len(result)} rows, After: {len(filtered)} rows")
filtered

---
## 7. eval() & query() for Performance

`eval()` and `query()` use numexpr under the hood, which can be faster than standard pandas operations for large DataFrames because it avoids creating intermediate arrays.

In [ ]:
n = 1_000_000
df_big = pd.DataFrame({
    'a': np.random.randn(n),
    'b': np.random.randn(n),
    'c': np.random.randn(n),
    'category': np.random.choice(['X', 'Y', 'Z'], n)
})

print(f"DataFrame shape: {df_big.shape}")
print(f"Memory: {df_big.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# eval — compute expressions efficiently
# Standard pandas: creates multiple intermediate arrays
%timeit df_big['a'] * df_big['b'] + df_big['c'] ** 2

# eval: single pass, less memory
%timeit df_big.eval('a * b + c ** 2')

In [ ]:
# eval can also create new columns with inplace=False (default)
df_big = df_big.eval("""
    d = a * b + c
    e = (a + b) / 2
    ratio = a / (b + 1)
""")
df_big.head()

In [ ]:
# query — filter rows using string expressions
# Supports @ prefix to reference local variables
threshold = 1.5

%timeit df_big[(df_big['a'] > threshold) & (df_big['b'] < -1) & (df_big['category'] == 'X')]
%timeit df_big.query('a > @threshold and b < -1 and category == "X"')

In [ ]:
# Complex query examples
result1 = df_big.query('a.abs() > 2 and (category == "X" or category == "Y")')
print(f"Complex query result: {len(result1)} rows")

# Using 'in' operator
cats = ['X', 'Z']
result2 = df_big.query('category in @cats and d > 0')
print(f"In-operator query result: {len(result2)} rows")

---
## 8. Categorical Data

Categorical dtype reduces memory usage for columns with limited unique values and enables ordering for custom sort logic.

In [ ]:
n = 500_000
df_cat = pd.DataFrame({
    'priority': np.random.choice(['Low', 'Medium', 'High', 'Critical'], n),
    'status': np.random.choice(['Open', 'In Progress', 'Resolved', 'Closed'], n),
    'value': np.random.randn(n)
})

print("Before conversion:")
print(f"Memory: {df_cat.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"priority dtype: {df_cat['priority'].dtype}")

In [ ]:
# Convert to ordered categorical
priority_order = pd.CategoricalDtype(
    categories=['Low', 'Medium', 'High', 'Critical'],
    ordered=True
)
df_cat['priority'] = df_cat['priority'].astype(priority_order)
df_cat['status'] = df_cat['status'].astype('category')

print("After conversion:")
print(f"Memory: {df_cat.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"priority dtype: {df_cat['priority'].dtype}")

In [ ]:
# Ordered categoricals enable comparison operators
high_priority = df_cat[df_cat['priority'] >= 'High']
print(f"Rows with priority >= High: {len(high_priority)}")

# sort_values respects categorical order
sample = df_cat.head(10).sort_values('priority')
sample[['priority', 'status', 'value']]

In [ ]:
# Categorical accessor — .cat
print("Categories:", df_cat['priority'].cat.categories.tolist())
print("Codes (integer representation):")
print(df_cat['priority'].cat.codes.head(10).values)

# Rename categories
df_cat['priority_abbr'] = df_cat['priority'].cat.rename_categories({
    'Low': 'L', 'Medium': 'M', 'High': 'H', 'Critical': 'C'
})
df_cat[['priority', 'priority_abbr']].head()

In [ ]:
# Performance benefit of categoricals in groupby
# groupby on categoricals uses observed=False by default to include all categories
print("GroupBy on categorical (includes all categories even if empty):")
df_cat.groupby('priority', observed=False)['value'].agg(['mean', 'count'])

---
## 9. Advanced DateTime Operations

pandas has rich datetime support through the `.dt` accessor, `pd.Timestamp`, and resampling.

In [ ]:
np.random.seed(42)
df_ts = pd.DataFrame({
    'timestamp': pd.date_range('2023-01-01', periods=730, freq='h'),
    'value': np.random.randn(730).cumsum() + 50,
    'category': np.random.choice(['A', 'B'], 730)
})
df_ts = df_ts.set_index('timestamp')
df_ts.head()

In [ ]:
# resample — time-based groupby
# Downsample to daily frequency
daily = df_ts['value'].resample('D').agg(['mean', 'min', 'max', 'std'])
print("Daily resampled:")
daily.head()

In [ ]:
# resample with custom aggregation per group
weekly_by_cat = df_ts.groupby('category').resample('W')['value'].mean()
print("Weekly mean by category:")
weekly_by_cat.head(10)

In [ ]:
# DateOffset — business day aware operations
from pandas.tseries.offsets import BDay, BMonthEnd, CustomBusinessDay

today = pd.Timestamp('2024-03-15')
print(f"Today: {today}")
print(f"5 business days later: {today + BDay(5)}")
print(f"End of business month: {today + BMonthEnd(0)}")
print(f"Next month end: {today + BMonthEnd(1)}")

In [ ]:
# Period and PeriodIndex — represent time spans
periods = pd.period_range('2024-01', periods=6, freq='M')
df_period = pd.DataFrame({
    'period': periods,
    'value': [100, 120, 115, 130, 125, 140]
})
print("Period DataFrame:")
print(df_period)
print(f"\nPeriod type: {type(df_period['period'][0])}")

# Convert between Timestamp and Period
print(f"\nPeriod to timestamp: {periods[0].to_timestamp()}")
print(f"Timestamp to period: {pd.Timestamp('2024-03-15').to_period('M')}")

In [ ]:
# Timezone handling
ts_utc = pd.Timestamp('2024-03-15 10:00', tz='UTC')
print(f"UTC:     {ts_utc}")
print(f"US/East: {ts_utc.tz_convert('US/Eastern')}")
print(f"Asia:    {ts_utc.tz_convert('Asia/Kolkata')}")

# Localize a naive timestamp
naive = pd.Timestamp('2024-03-15 10:00')
localized = naive.tz_localize('Europe/London')
print(f"\nLocalized: {localized}")

### Exercise 9
1. Resample `df_ts` to 6-hour intervals and compute the standard deviation of `value`.
2. Create a new column `hour_of_day` from the index and compute the average value per hour.
3. Use `between_time('09:00', '17:00')` to filter business hours only.

In [ ]:
# Solution 9.1: Resample to 6-hour intervals, compute std of value
resampled_6h = df_ts['value'].resample('6h').std()
print("1. 6-hour resampled standard deviation:")
print(resampled_6h.head(10))

# Solution 9.2: Average value per hour of day
df_ts_copy = df_ts.copy()
df_ts_copy['hour_of_day'] = df_ts_copy.index.hour
avg_by_hour = df_ts_copy.groupby('hour_of_day')['value'].mean()
print("\n2. Average value per hour of day:")
print(avg_by_hour)

# Solution 9.3: Filter business hours only (09:00 to 17:00)
business_hours = df_ts.between_time('09:00', '17:00')
print(f"\n3. Total rows: {len(df_ts)}, Business hours only: {len(business_hours)}")
print(business_hours.head(10))


---
## 10. Advanced String Operations

The `.str` accessor provides vectorized string operations. Here we go beyond basics into regex, extraction, and complex transformations.

In [ ]:
df_str = pd.DataFrame({
    'raw_text': [
        'John Doe <john.doe@email.com> (Manager)',
        'Jane Smith <jsmith@corp.org> (Director)',
        'Bob Johnson <bob.j@test.net> (Analyst)',
        'Alice Brown <alice.b@company.io> (VP)',
        'Charlie <charlie@example.com> (Intern)'
    ],
    'address': [
        '123 Main St, New York, NY 10001',
        '456 Oak Ave, Los Angeles, CA 90001',
        '789 Pine Rd, Chicago, IL 60601',
        '321 Elm Blvd, Houston, TX 77001',
        '654 Maple Dr, Phoenix, AZ 85001'
    ]
})

df_str

In [ ]:
# str.extract — capture groups with regex
# Extract name, email, and title in one shot
extracted = df_str['raw_text'].str.extract(
    r'^(?P<name>[\w\s]+?)\s*<(?P<email>[^>]+)>\s*\((?P<title>\w+)\)'
)
extracted

In [ ]:
# str.extractall — extract ALL matches (returns MultiIndex)
text_series = pd.Series([
    'Order #123 and #456 shipped',
    'Order #789 pending',
    'No orders today',
    'Orders #101, #202, #303 delivered'
])

all_orders = text_series.str.extractall(r'#(\d+)')
all_orders.columns = ['order_id']
print("All extracted matches:")
all_orders

In [ ]:
# Parse structured address data
address_parts = df_str['address'].str.extract(
    r'^(?P<street>[\d\w\s]+),\s*(?P<city>[\w\s]+),\s*(?P<state>[A-Z]{2})\s+(?P<zip>\d{5})'
)
address_parts

In [ ]:
# str.replace with regex — clean up text
messy = pd.Series(['  Hello   World  ', 'This\t\thas\ttabs', 'Multiple\n\nNewlines'])

# Replace any whitespace sequence with a single space
cleaned = messy.str.replace(r'\s+', ' ', regex=True).str.strip()
print("Cleaned:")
print(cleaned.values)

In [ ]:
# str.split with expand — split into columns
names = pd.Series(['John-Michael-Doe', 'Jane-Ann-Smith', 'Bob-Lee-Johnson'])
split_names = names.str.split('-', expand=True)
split_names.columns = ['first', 'middle', 'last']
split_names

---
## 11. Memory Optimization

Reducing memory footprint is critical when working with large datasets.

In [ ]:
def optimize_dtypes(df):
    """Downcast numeric columns and convert low-cardinality strings to categorical."""
    result = df.copy()
    
    for col in result.select_dtypes(include=['int']).columns:
        result[col] = pd.to_numeric(result[col], downcast='integer')
    
    for col in result.select_dtypes(include=['float']).columns:
        result[col] = pd.to_numeric(result[col], downcast='float')
    
    for col in result.select_dtypes(include=['object']).columns:
        ratio = result[col].nunique() / len(result)
        if ratio < 0.5:  # less than 50% unique values
            result[col] = result[col].astype('category')
    
    return result

In [ ]:
n = 500_000
df_large = pd.DataFrame({
    'id': np.arange(n),
    'value_int': np.random.randint(0, 100, n),
    'value_float': np.random.randn(n),
    'status': np.random.choice(['active', 'inactive', 'pending'], n),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
})

mem_before = df_large.memory_usage(deep=True).sum() / 1e6
df_optimized = optimize_dtypes(df_large)
mem_after = df_optimized.memory_usage(deep=True).sum() / 1e6

print(f"Before: {mem_before:.2f} MB")
print(f"After:  {mem_after:.2f} MB")
print(f"Savings: {(1 - mem_after/mem_before)*100:.1f}%")
print("\nDtype changes:")
for col in df_large.columns:
    if df_large[col].dtype != df_optimized[col].dtype:
        print(f"  {col}: {df_large[col].dtype} → {df_optimized[col].dtype}")

In [ ]:
# Reading large files in chunks
# Useful when data doesn't fit in memory

# Simulating chunk processing (we create a temp CSV first)
import tempfile, os

temp_path = os.path.join(tempfile.gettempdir(), 'large_data.csv')
df_large.to_csv(temp_path, index=False)

chunk_results = []
for chunk in pd.read_csv(temp_path, chunksize=100_000):
    chunk_agg = chunk.groupby('region')['value_float'].mean()
    chunk_results.append(chunk_agg)

# Combine chunk results
final = pd.concat(chunk_results, axis=1).mean(axis=1)
print("Chunk-processed regional means:")
print(final)

os.remove(temp_path)

---
## 12. Vectorization & Performance Patterns

Knowing when to use vectorized operations vs. apply vs. iterrows is critical for performance.

In [ ]:
n = 100_000
df_perf = pd.DataFrame({
    'a': np.random.randn(n),
    'b': np.random.randn(n),
    'category': np.random.choice(['X', 'Y', 'Z'], n)
})

In [ ]:
# SLOW: iterrows (Python-level loop)
%%timeit -n1 -r1
results = []
for idx, row in df_perf.iterrows():
    results.append(row['a'] ** 2 + row['b'] ** 2)

In [ ]:
# MEDIUM: apply (slightly faster than iterrows but still row-level Python)
%%timeit -n5 -r3
df_perf.apply(lambda row: row['a'] ** 2 + row['b'] ** 2, axis=1)

In [ ]:
# FAST: vectorized numpy operations
%%timeit -n50 -r5
df_perf['a'] ** 2 + df_perf['b'] ** 2

In [ ]:
# FASTEST: .values (raw numpy arrays) for pure computation
%%timeit -n50 -r5
df_perf['a'].values ** 2 + df_perf['b'].values ** 2

In [ ]:
# np.where — vectorized conditional logic (replaces if/else in loops)
df_perf['label'] = np.where(
    df_perf['a'] > 0,
    'positive',
    'non-positive'
)

# np.select — vectorized multi-condition logic
conditions = [
    (df_perf['a'] > 1) & (df_perf['b'] > 1),
    (df_perf['a'] > 1) & (df_perf['b'] <= 1),
    (df_perf['a'] <= 1) & (df_perf['b'] > 1)
]
choices = ['both_high', 'a_high', 'b_high']
df_perf['quadrant'] = np.select(conditions, choices, default='both_low')

df_perf['quadrant'].value_counts()

In [ ]:
# map vs apply for column-level transformations
mapping = {'X': 'Category_X', 'Y': 'Category_Y', 'Z': 'Category_Z'}

# map — use for element-wise mappings from dict or function
%timeit df_perf['category'].map(mapping)

# replace — similar but more flexible
%timeit df_perf['category'].replace(mapping)

### Performance Summary

| Method | Speed | Use Case |
|--------|-------|----------|
| `.values` + numpy | Fastest | Pure numeric computation |
| Vectorized pandas | Fast | Standard column operations |
| `eval()` / `query()` | Fast | Complex expressions on large DataFrames |
| `.map()` | Fast | Element-wise dict/function mapping |
| `np.where` / `np.select` | Fast | Conditional logic |
| `.apply(axis=1)` | Slow | Complex row-level logic (last resort) |
| `iterrows()` | Slowest | Avoid — almost never needed |

---
## 13. Custom Accessors

You can extend pandas with custom accessors using `@pd.api.extensions.register_dataframe_accessor`. This lets you add domain-specific methods to any DataFrame.

In [ ]:
@pd.api.extensions.register_dataframe_accessor("stats")
class StatsAccessor:
    def __init__(self, pandas_obj):
        self._obj = pandas_obj

    def summary(self):
        """Enhanced describe with additional stats."""
        df = self._obj.select_dtypes(include='number')
        result = df.describe()
        result.loc['skew'] = df.skew()
        result.loc['kurtosis'] = df.kurtosis()
        result.loc['iqr'] = df.quantile(0.75) - df.quantile(0.25)
        result.loc['cv'] = df.std() / df.mean()  # coefficient of variation
        return result

    def outlier_report(self, method='iqr', threshold=1.5):
        """Report outliers per numeric column."""
        df = self._obj.select_dtypes(include='number')
        report = {}
        for col in df.columns:
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - threshold * iqr
            upper = q3 + threshold * iqr
            n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            report[col] = {
                'n_outliers': n_outliers,
                'pct_outliers': round(n_outliers / len(df) * 100, 2),
                'lower_bound': round(lower, 2),
                'upper_bound': round(upper, 2)
            }
        return pd.DataFrame(report).T

In [ ]:
sample_df = pd.DataFrame({
    'revenue': np.random.lognormal(10, 1, 1000),
    'quantity': np.random.poisson(20, 1000),
    'discount': np.random.beta(2, 5, 1000)
})

print("Enhanced summary:")
print(sample_df.stats.summary())

print("\nOutlier report:")
print(sample_df.stats.outlier_report())

---
## 14. Styling DataFrames

The `.style` API creates visually rich HTML tables in Jupyter. Useful for reports and presentations.

In [ ]:
np.random.seed(42)
df_style = pd.DataFrame({
    'Region': ['North', 'South', 'East', 'West'],
    'Revenue': [1250000, 980000, 1100000, 870000],
    'Profit': [250000, -50000, 180000, 120000],
    'Growth_%': [12.5, -3.2, 8.7, 5.1],
    'Margin_%': [20.0, -5.1, 16.4, 13.8]
}).set_index('Region')

In [ ]:
# Background gradient — heat map style
df_style.style.background_gradient(cmap='RdYlGn', subset=['Revenue', 'Profit'])

In [ ]:
# Conditional formatting — highlight negatives in red
def highlight_negative(val):
    color = 'color: red; font-weight: bold' if val < 0 else ''
    return color

(
    df_style.style
    .map(highlight_negative, subset=['Profit', 'Growth_%', 'Margin_%'])
    .format({
        'Revenue': '${:,.0f}',
        'Profit': '${:,.0f}',
        'Growth_%': '{:+.1f}%',
        'Margin_%': '{:+.1f}%'
    })
    .bar(subset=['Revenue'], color='#5fba7d', vmin=0)
    .set_caption('Regional Performance Report')
)

In [ ]:
# Highlight min/max
(
    df_style.style
    .highlight_max(color='lightgreen', subset=['Revenue', 'Profit'])
    .highlight_min(color='#ffcccc', subset=['Revenue', 'Profit'])
    .format({'Revenue': '${:,.0f}', 'Profit': '${:,.0f}'})
)

In [ ]:
# Complex styling with set_properties and custom CSS
def color_growth(val):
    if val > 10:
        return 'background-color: #28a745; color: white'
    elif val > 0:
        return 'background-color: #d4edda'
    elif val > -5:
        return 'background-color: #f8d7da'
    else:
        return 'background-color: #dc3545; color: white'

(
    df_style.style
    .map(color_growth, subset=['Growth_%'])
    .format({
        'Revenue': '${:,.0f}',
        'Profit': '${:,.0f}',
        'Growth_%': '{:+.1f}%',
        'Margin_%': '{:+.1f}%'
    })
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#343a40'), ('color', 'white')]},
        {'selector': 'td', 'props': [('text-align', 'right')]}
    ])
    .set_caption('Styled Regional Report')
)

---
## Bonus: Useful Tricks & One-Liners

A collection of lesser-known pandas patterns.

In [ ]:
# 1. nlargest / nsmallest — faster than sort + head
df_sales.nlargest(5, 'revenue')[['region', 'product', 'revenue']]

In [ ]:
# 2. clip — cap values to a range
s = pd.Series([-5, -2, 0, 3, 10, 100])
print("Original:", s.values)
print("Clipped [0, 10]:", s.clip(lower=0, upper=10).values)

In [ ]:
# 3. explode — unnest lists into rows
df_tags = pd.DataFrame({
    'item': ['A', 'B', 'C'],
    'tags': [['red', 'blue'], ['green'], ['red', 'green', 'blue']]
})
print("Before explode:")
print(df_tags)
print("\nAfter explode:")
print(df_tags.explode('tags'))

In [ ]:
# 4. factorize — encode labels as integers (useful for ML preprocessing)
codes, uniques = pd.factorize(['cat', 'dog', 'cat', 'bird', 'dog', 'cat'])
print(f"Codes:   {codes}")
print(f"Uniques: {uniques}")

In [ ]:
# 5. duplicated with keep — find first, last, or all duplicates
df_dup = pd.DataFrame({'A': [1, 1, 2, 3, 3], 'B': ['a', 'a', 'b', 'c', 'c']})
print("keep='first' (default — marks later dupes):")
print(df_dup.duplicated().values)
print("keep=False (marks ALL duplicates):")
print(df_dup.duplicated(keep=False).values)

In [ ]:
# 6. at / iat — fastest scalar access
df_small = pd.DataFrame({'x': [10, 20, 30]}, index=['a', 'b', 'c'])
print(f"at['b', 'x']: {df_small.at['b', 'x']}")
print(f"iat[1, 0]:    {df_small.iat[1, 0]}")

In [ ]:
# 7. where / mask — conditional replacement
s = pd.Series([1, 2, 3, 4, 5])
print("where (keep where True, else NaN):", s.where(s > 3).values)
print("mask (replace where True with NaN):", s.mask(s > 3).values)
print("where with fill:", s.where(s > 3, other=-1).values)

In [ ]:
# 8. Sparse arrays — memory-efficient for data with many repeated values
sparse_data = pd.arrays.SparseArray([0, 0, 0, 1, 0, 0, 0, 2, 0, 0])
print(f"Dense memory:  {sparse_data.nbytes} bytes")
print(f"Sparse memory: {sparse_data.sp_values.nbytes + sparse_data.sp_index.nbytes} bytes")
print(f"Density: {sparse_data.density}")

In [ ]:
# 9. json_normalize — flatten nested JSON/dicts
from pandas import json_normalize

nested_data = [
    {'name': 'Alice', 'address': {'city': 'NYC', 'state': 'NY'}, 'scores': [85, 90]},
    {'name': 'Bob', 'address': {'city': 'LA', 'state': 'CA'}, 'scores': [78, 88]}
]
flat = json_normalize(nested_data)
flat

In [ ]:
# 10. compare — show differences between two DataFrames
df_v1 = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df_v2 = pd.DataFrame({'A': [1, 20, 3], 'B': [4, 5, 60]})

diff = df_v1.compare(df_v2)
print("Differences between v1 and v2:")
diff

---
## Final Challenge

Put it all together! Using the `df_sales` DataFrame created in Section 2:

1. Create a method chain that:
   - Filters out rows where revenue is negative
   - Adds a `month` column from the `date` column
   - Creates `revenue_per_unit = revenue / units`
   - Converts `region` to an ordered categorical

2. Build a pivot table showing mean `revenue_per_unit` by `region` (rows) and `month` (columns).

3. Style the pivot table with:
   - Background gradient
   - Format to 2 decimal places with `$` prefix
   - Highlight the max value in each row

4. For each region, compute a 7-day rolling mean of revenue (sort by date first).

5. Compare the performance of:
   - `apply` with a lambda for `revenue * units`
   - Vectorized pandas multiplication
   - Raw numpy multiplication with `.values`

In [ ]:
# Final Challenge Solutions

# ---------- 1. Method chain ----------
region_order = pd.CategoricalDtype(
    categories=['North', 'East', 'South', 'West'], ordered=True
)

df_challenge = (
    df_sales
    .query('revenue > 0')
    .assign(
        month=lambda df: df['date'].dt.to_period('M'),
        revenue_per_unit=lambda df: df['revenue'] / df['units'],
        region=lambda df: df['region'].astype(region_order)
    )
)
print("1. Method-chained DataFrame:")
print(df_challenge[['region', 'month', 'revenue', 'units', 'revenue_per_unit']].head(10))

# ---------- 2. Pivot table ----------
pivot_rpu = df_challenge.pivot_table(
    values='revenue_per_unit',
    index='region',
    columns='month',
    aggfunc='mean'
)
print("\n2. Pivot table (mean revenue per unit):")
print(pivot_rpu)

# ---------- 3. Styled pivot table ----------
print("\n3. Styled pivot table (displays in Jupyter):")
styled = (
    pivot_rpu.style
    .background_gradient(cmap='YlGnBu', axis=None)
    .format('${:.2f}')
    .highlight_max(color='lightgreen', axis=1)
    .set_caption('Mean Revenue Per Unit by Region & Month')
)
styled

# ---------- 4. 7-day rolling mean per region ----------
df_rolling = df_sales.sort_values(['region', 'date']).copy()
df_rolling['rolling_7d_revenue'] = (
    df_rolling.groupby('region')['revenue']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
print("\n4. 7-day rolling mean revenue per region:")
print(df_rolling[['region', 'date', 'revenue', 'rolling_7d_revenue']].head(15))

# ---------- 5. Performance comparison ----------
import time

# apply
start = time.perf_counter()
for _ in range(10):
    df_sales.apply(lambda row: row['revenue'] * row['units'], axis=1)
t_apply = (time.perf_counter() - start) / 10

# Vectorized pandas
start = time.perf_counter()
for _ in range(1000):
    df_sales['revenue'] * df_sales['units']
t_pandas = (time.perf_counter() - start) / 1000

# Raw numpy
start = time.perf_counter()
for _ in range(1000):
    df_sales['revenue'].values * df_sales['units'].values
t_numpy = (time.perf_counter() - start) / 1000

print("\n5. Performance comparison (revenue × units):")
print(f"   apply:            {t_apply*1000:.3f} ms")
print(f"   vectorized pandas: {t_pandas*1000:.3f} ms  ({t_apply/t_pandas:.0f}× faster than apply)")
print(f"   raw numpy:        {t_numpy*1000:.3f} ms  ({t_apply/t_numpy:.0f}× faster than apply)")
